# Задача 1 — SFT: перенос простого стиля

Дообучаем `Qwen/Qwen3-4B-Instruct-2507` через QLoRA на датасете `kid_adult`. Цель — перенести простой стиль ответа и измерить средний `P_simple` на `public_test_style`.


### Зависимости

In [ ]:
%pip install -q -U "transformers>=4.53.0" "accelerate>=1.8.0" "peft>=0.16.0" "bitsandbytes>=0.46.0" scipy scikit-learn safetensors tqdm


### Импорты и сиды
Фиксируем всё для воспроизводимости. tf32 на амперах ускоряет матричные умножения без заметной потери точности.

In [ ]:
import gc, json, os, random
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np, torch
from scipy.sparse import hstack
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM, AutoModelForSequenceClassification, AutoTokenizer, BitsAndBytesConfig,
    get_linear_schedule_with_warmup, set_seed
)
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training

SEED = 42
MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
DATA_DIR = Path("task_data")
STYLE_CLF_PATH = DATA_DIR / "style_clf.pkl"
ADAPTER_DIR = Path("adapters/task1_sft")
os.environ["HF_HUB_DISABLE_XET"] = "1"

def fix_seeds(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)
    torch.backends.cuda.matmul.allow_tf32 = True

fix_seeds()
print("Seed:", SEED)
print("CUDA available:", torch.cuda.is_available())


### Данные
Скачиваем с GitHub, если локально нет.

In [ ]:
GITHUB_RAW_BASE = "https://raw.githubusercontent.com/Dom82209/ml-olympiad-red-exp/main/task_data"

def download_if_missing(filename):
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    path = DATA_DIR / filename
    if path.exists():
        return path
    url = f"{GITHUB_RAW_BASE}/{filename}"
    print(f"Downloading {filename}...")
    urlretrieve(url, path)
    return path

def read_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

download_if_missing("kid_adult.jsonl")
download_if_missing("public_test_style.jsonl")
download_if_missing("style_clf.pkl")
download_if_missing("good_bad.jsonl")
download_if_missing("public_test_quality.jsonl")

train_data = read_jsonl(DATA_DIR / "kid_adult.jsonl")
test_data = read_jsonl(DATA_DIR / "public_test_style.jsonl")
good_bad_data = read_jsonl(DATA_DIR / "good_bad.jsonl")
quality_test_data = read_jsonl(DATA_DIR / "public_test_quality.jsonl")

print(f"Loaded style: train={len(train_data)}, test={len(test_data)}")
print(f"Loaded quality: train={len(good_bad_data)}, test={len(quality_test_data)}")
print("Style example keys:", train_data[0].keys())
print("Quality example keys:", good_bad_data[0].keys())


### Токенайзер
Настраиваем чат-шаблон. padding_side="right" — обязательно для causal LM, иначе attention mask ломается.

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "right"

def fmt_prompt(p):
    return tok.apply_chat_template(
        [{"role": "user", "content": p}],
        tokenize=False, add_generation_prompt=True
    )

def fmt_full(p, a):
    return tok.apply_chat_template(
        [{"role": "user", "content": p}, {"role": "assistant", "content": a}],
        tokenize=False, add_generation_prompt=False
    )

### Датасет
Маскируем промпт (-100), лосс только по ответу.

In [ ]:
class SFTDataset(Dataset):
    def __init__(self, rows, tok, max_len=512):
        self.items = []
        for row in rows:
            p_ids = tok(fmt_prompt(row["prompt"]), add_special_tokens=False, truncation=True, max_length=max_len)["input_ids"]
            full_ids = tok(fmt_full(row["prompt"], row["kid"]), add_special_tokens=False, truncation=True, max_length=max_len)["input_ids"]

            labels = full_ids.copy()
            p_len = min(len(p_ids), len(labels))
            labels[:p_len] = [-100] * p_len # игнорим промпт при лоссе

            self.items.append({"input_ids": full_ids, "labels": labels})

    def __len__(self): return len(self.items)
    def __getitem__(self, idx): return self.items[idx]

def collate_fn(batch):
    max_len = max(len(x["input_ids"]) for x in batch)
    inp, mask, lbl = [], [], []
    for x in batch:
        pad = max_len - len(x["input_ids"])
        inp.append(x["input_ids"] + [tok.pad_token_id] * pad)
        mask.append([1] * len(x["input_ids"]) + [0] * pad)
        lbl.append(x["labels"] + [-100] * pad)

    return {
        "input_ids": torch.tensor(inp),
        "attention_mask": torch.tensor(mask),
        "labels": torch.tensor(lbl)
    }

dataset = SFTDataset(train_data, tok)
print("Dataset size:", len(dataset))

### Модель и LoRA
Загружаем в 4-битном кванте (nf4), навешиваем LoRA-адаптеры на все линейные проекции. use_cache=False — без этого градиенты считаются некорректно при обучении.

In [ ]:
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True
)

lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.0, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_cfg, device_map="auto", trust_remote_code=True
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

### Обучение
Ручной трейн-луп. grad_acc=8 чтобы эффективный батч был 8 при реальном bs=1 (иначе в 24GB VRAM не влезает). Сохраняем только адаптеры — base-модель не трогаем.

In [ ]:
fix_seeds()
epochs, bs, grad_acc, lr = 1, 1, 8, 2e-4

loader = DataLoader(dataset, batch_size=bs, shuffle=True, collate_fn=collate_fn)
opt = torch.optim.AdamW(model.parameters(), lr=lr)
total_steps = max(1, epochs * len(loader) // grad_acc)
sched = get_linear_schedule_with_warmup(opt, num_warmup_steps=max(1, int(total_steps * 0.03)), num_training_steps=total_steps)

model.train()
opt.zero_grad(set_to_none=True)

for epoch in range(epochs):
    pbar = tqdm(loader, desc=f"Epoch {epoch+1}")
    for step, batch in enumerate(pbar):
        batch = {k: v.to(model.device) for k, v in batch.items()}
        loss = model(**batch).loss / grad_acc
        loss.backward()

        if (step + 1) % grad_acc == 0:
            opt.step()
            sched.step()
            opt.zero_grad(set_to_none=True)

        pbar.set_postfix(loss=float(loss.detach().cpu()) * grad_acc)

ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(ADAPTER_DIR)
print("Saved adapters to", ADAPTER_DIR)

### Генерация на тесте
do_sample=False — по условию задачи нужна детерминированная генерация. При декодировании обрезаем входной промпт, оставляем только ответ модели.

In [ ]:
@torch.no_grad()
def generate(p, max_new=160):
    model.eval()
    inputs = tok(fmt_prompt(p), return_tensors="pt", add_special_tokens=False).to(model.device)
    out = model.generate(
        **inputs, max_new_tokens=max_new, do_sample=False,
        pad_token_id=tok.eos_token_id, eos_token_id=tok.eos_token_id
    )
    return tok.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

answers = [generate(row["prompt"]) for row in tqdm(test_data, desc="Generating")]
print("Generated:", len(answers))
print("Example:", answers[0][:100])

### Метрика
Считаем P_simple через предобученный клф. Пороги для буквы — из условия.

In [ ]:
import pickle

with open(STYLE_CLF_PATH, "rb") as f:
    obj = pickle.load(f)

clf, (w_vec, c_vec) = obj["clf"], obj["vecs"]
simple_idx = list(clf.classes_).index(1)

def get_p_simple(texts):
    feats = hstack([w_vec.transform(texts), c_vec.transform(texts)])
    return clf.predict_proba(feats)[:, simple_idx]

scores = get_p_simple(answers)
metric = float(np.mean(scores))

print(f"Task 1 SFT P_simple = {metric:.6f}")
print(f"Min P_simple = {np.min(scores):.6f}, max = {np.max(scores):.6f}")

def get_letter(v):
    if v < 0.4: return "А"
    if v < 0.7: return "Б"
    if v < 0.9: return "В"
    return "Г"

print(f"Task 1 answer interval letter = {get_letter(metric)}")

### Задача 2 — DPO по стилю
DPO запускается поверх SFT-модели из задачи 1. В датасете `kid_adult` простой ответ `kid` используется как `chosen`, а сложный ответ `adult` — как `rejected`.


In [ ]:
def answer_logprob(model, prompt, answer, average=False, max_len=768):
    prompt_ids = tok(
        fmt_prompt(prompt),
        return_tensors="pt",
        add_special_tokens=False,
        truncation=True,
        max_length=max_len,
    )["input_ids"]

    full = tok(
        fmt_full(prompt, answer),
        return_tensors="pt",
        add_special_tokens=False,
        truncation=True,
        max_length=max_len,
    )

    input_ids = full["input_ids"].to(model.device)
    attention_mask = full["attention_mask"].to(model.device)
    prompt_len = min(prompt_ids.shape[1], input_ids.shape[1])

    labels = input_ids.clone()
    labels[:, :prompt_len] = -100
    labels[attention_mask == 0] = -100

    out = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = out.logits[:, :-1, :]
    targets = input_ids[:, 1:]
    mask = labels[:, 1:] != -100

    log_probs = torch.log_softmax(logits, dim=-1)
    token_log_probs = log_probs.gather(-1, targets.unsqueeze(-1)).squeeze(-1)
    selected = token_log_probs[mask]

    if selected.numel() == 0:
        return torch.tensor(-100.0, device=model.device)
    return selected.mean() if average else selected.sum()

@torch.no_grad()
def precompute_reference_logps(rows):
    model.eval()
    ref_logps = []
    for row in tqdm(rows, desc="Reference logprobs"):
        chosen = answer_logprob(model, row["prompt"], row["kid"], average=False)
        rejected = answer_logprob(model, row["prompt"], row["adult"], average=False)
        ref_logps.append((float(chosen.cpu()), float(rejected.cpu())))
    return ref_logps


### Обучение DPO по стилю
Reference logprob считается один раз на SFT-модели до обновлений. Затем модель обучается предпочитать `kid` перед `adult`.


In [ ]:
fix_seeds()

dpo_rows = train_data
ref_logps = precompute_reference_logps(dpo_rows)

dpo_epochs = 1
dpo_grad_acc = 8
dpo_lr = 5e-5
beta = 0.1

opt = torch.optim.AdamW(model.parameters(), lr=dpo_lr)
total_steps = max(1, dpo_epochs * len(dpo_rows) // dpo_grad_acc)
sched = get_linear_schedule_with_warmup(
    opt,
    num_warmup_steps=max(1, int(total_steps * 0.03)),
    num_training_steps=total_steps,
)

model.train()
opt.zero_grad(set_to_none=True)
indices = np.arange(len(dpo_rows))

for epoch in range(dpo_epochs):
    np.random.shuffle(indices)
    pbar = tqdm(indices, desc=f"DPO epoch {epoch + 1}")
    for step, idx in enumerate(pbar):
        row = dpo_rows[int(idx)]
        ref_chosen, ref_rejected = ref_logps[int(idx)]

        pi_chosen = answer_logprob(model, row["prompt"], row["kid"], average=False)
        pi_rejected = answer_logprob(model, row["prompt"], row["adult"], average=False)

        pi_logratio = pi_chosen - pi_rejected
        ref_logratio = torch.tensor(ref_chosen - ref_rejected, device=model.device)
        loss = -torch.nn.functional.logsigmoid(beta * (pi_logratio - ref_logratio)) / dpo_grad_acc
        loss.backward()

        if (step + 1) % dpo_grad_acc == 0:
            opt.step()
            sched.step()
            opt.zero_grad(set_to_none=True)

        pbar.set_postfix(loss=float(loss.detach().cpu()) * dpo_grad_acc)

DPO_ADAPTER_DIR = Path("adapters/task2_dpo_style")
DPO_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(DPO_ADAPTER_DIR)
print("Saved DPO adapter to", DPO_ADAPTER_DIR)


### Метрика задачи 2
После DPO снова генерируем ответы на `public_test_style` с greedy decoding и считаем средний `P_simple`.


In [ ]:
dpo_answers = [generate(row["prompt"]) for row in tqdm(test_data, desc="Generating after DPO")]
dpo_scores = get_p_simple(dpo_answers)
task2_metric = float(np.mean(dpo_scores))

print(f"Task 2 Style DPO P_simple = {task2_metric:.6f}")
print(f"Min P_simple = {np.min(dpo_scores):.6f}, max = {np.max(dpo_scores):.6f}")
print(f"Task 2 answer interval letter = {get_letter(task2_metric)}")


### Освобождение памяти перед Reward Model
Reward Model загружается как отдельная модель-оценщик, поэтому после задач 1-2 освобождаем память от causal LM.


In [ ]:
del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Causal LM removed from memory")


### Задача 3 — Reward Model
Обучаем модель-оценщик на `good_bad`: хороший ответ `chosen` должен получать reward выше, чем плохой ответ `rejected`.


In [ ]:
rm_lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",
    task_type="SEQ_CLS",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    modules_to_save=["score"],
)

rm_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1,
    quantization_config=bnb_cfg,
    device_map="auto",
    trust_remote_code=True,
)
rm_model.config.pad_token_id = tok.pad_token_id
rm_model.config.use_cache = False
rm_model = prepare_model_for_kbit_training(rm_model)
rm_model = get_peft_model(rm_model, rm_lora_cfg)
rm_model.print_trainable_parameters()

def reward_score(model, prompt, answer, max_len=768):
    text = fmt_full(prompt, answer)
    batch = tok(
        text,
        return_tensors="pt",
        add_special_tokens=False,
        truncation=True,
        max_length=max_len,
    ).to(model.device)
    return model(**batch).logits.squeeze(-1)


### Обучение Reward Model
Оптимизируем pairwise loss: `chosen` должен иметь больший reward, чем `rejected`.


In [ ]:
fix_seeds()

rm_epochs = 1
rm_grad_acc = 8
rm_lr = 1e-4

opt = torch.optim.AdamW(rm_model.parameters(), lr=rm_lr)
total_steps = max(1, rm_epochs * len(good_bad_data) // rm_grad_acc)
sched = get_linear_schedule_with_warmup(
    opt,
    num_warmup_steps=max(1, int(total_steps * 0.03)),
    num_training_steps=total_steps,
)

rm_model.train()
opt.zero_grad(set_to_none=True)
indices = np.arange(len(good_bad_data))

for epoch in range(rm_epochs):
    np.random.shuffle(indices)
    pbar = tqdm(indices, desc=f"Reward Model epoch {epoch + 1}")
    for step, idx in enumerate(pbar):
        row = good_bad_data[int(idx)]
        chosen_reward = reward_score(rm_model, row["instruction"], row["chosen"])
        rejected_reward = reward_score(rm_model, row["instruction"], row["rejected"])
        loss = -torch.nn.functional.logsigmoid(chosen_reward - rejected_reward).mean() / rm_grad_acc
        loss.backward()

        if (step + 1) % rm_grad_acc == 0:
            opt.step()
            sched.step()
            opt.zero_grad(set_to_none=True)

        pbar.set_postfix(loss=float(loss.detach().cpu()) * rm_grad_acc)

RM_ADAPTER_DIR = Path("adapters/task3_reward_model")
RM_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
rm_model.save_pretrained(RM_ADAPTER_DIR)
print("Saved reward model adapter to", RM_ADAPTER_DIR)


### Метрика задачи 3
Pairwise accuracy на `public_test_quality`: доля пар, где reward для `chosen` выше, чем для `rejected`.


In [ ]:
@torch.no_grad()
def reward_pairwise_accuracy(model, rows):
    model.eval()
    correct = 0
    for row in tqdm(rows, desc="Reward Model accuracy"):
        prompt = row.get("prompt") or row.get("instruction")
        chosen = reward_score(model, prompt, row["chosen"])
        rejected = reward_score(model, prompt, row["rejected"])
        correct += int(float(chosen.cpu()) > float(rejected.cpu()))
    return correct / len(rows)

task3_metric = reward_pairwise_accuracy(rm_model, quality_test_data)

def get_quality_letter(v):
    if v < 0.6: return "А"
    if v < 0.75: return "Б"
    if v < 0.9: return "В"
    return "Г"

print(f"Task 3 Reward Model pairwise accuracy = {task3_metric:.6f}")
print(f"Task 3 answer interval letter = {get_quality_letter(task3_metric)}")


### Освобождение памяти перед DPO по качеству
После оценки Reward Model освобождаем память и снова загружаем causal LM для DPO по качеству.


In [ ]:
try:
    del rm_model
except NameError:
    pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Reward Model removed from memory")


### Задача 4 — DPO по качеству
Загружаем базовую causal LM со style-DPO адаптером из задачи 2 и продолжаем DPO на `good_bad`, где `chosen` — хороший ответ, `rejected` — плохой.


In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_cfg,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)
model = PeftModel.from_pretrained(model, DPO_ADAPTER_DIR, is_trainable=True)
model.print_trainable_parameters()

def quality_prompt(row):
    return row.get("prompt") or row.get("instruction")

@torch.no_grad()
def precompute_quality_reference_logps(rows):
    model.eval()
    ref_logps = []
    for row in tqdm(rows, desc="Quality reference logprobs"):
        prompt = quality_prompt(row)
        chosen = answer_logprob(model, prompt, row["chosen"], average=False)
        rejected = answer_logprob(model, prompt, row["rejected"], average=False)
        ref_logps.append((float(chosen.cpu()), float(rejected.cpu())))
    return ref_logps


### Обучение DPO по качеству
DPO обучается на парах `good_bad`: `chosen` должен быть предпочтительнее `rejected`. Для обучения используется DPO loss с reference logprob стартовой style-модели.


In [ ]:
fix_seeds()

quality_ref_logps = precompute_quality_reference_logps(good_bad_data)

quality_dpo_epochs = 1
quality_dpo_grad_acc = 8
quality_dpo_lr = 5e-5
quality_beta = 0.1

opt = torch.optim.AdamW(model.parameters(), lr=quality_dpo_lr)
total_steps = max(1, quality_dpo_epochs * len(good_bad_data) // quality_dpo_grad_acc)
sched = get_linear_schedule_with_warmup(
    opt,
    num_warmup_steps=max(1, int(total_steps * 0.03)),
    num_training_steps=total_steps,
)

model.train()
opt.zero_grad(set_to_none=True)
indices = np.arange(len(good_bad_data))

for epoch in range(quality_dpo_epochs):
    np.random.shuffle(indices)
    pbar = tqdm(indices, desc=f"Quality DPO epoch {epoch + 1}")
    for step, idx in enumerate(pbar):
        row = good_bad_data[int(idx)]
        prompt = quality_prompt(row)
        ref_chosen, ref_rejected = quality_ref_logps[int(idx)]

        pi_chosen = answer_logprob(model, prompt, row["chosen"], average=False)
        pi_rejected = answer_logprob(model, prompt, row["rejected"], average=False)

        pi_logratio = pi_chosen - pi_rejected
        ref_logratio = torch.tensor(ref_chosen - ref_rejected, device=model.device)
        loss = -torch.nn.functional.logsigmoid(quality_beta * (pi_logratio - ref_logratio)) / quality_dpo_grad_acc
        loss.backward()

        if (step + 1) % quality_dpo_grad_acc == 0:
            opt.step()
            sched.step()
            opt.zero_grad(set_to_none=True)

        pbar.set_postfix(loss=float(loss.detach().cpu()) * quality_dpo_grad_acc)

QUALITY_DPO_ADAPTER_DIR = Path("adapters/task4_dpo_quality")
QUALITY_DPO_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(QUALITY_DPO_ADAPTER_DIR)
print("Saved quality DPO adapter to", QUALITY_DPO_ADAPTER_DIR)


### Метрика задачи 4
Implicit-preference accuracy на `public_test_quality`: сравниваем средний logprob на токен ответа. Генерация не используется.


In [ ]:
@torch.no_grad()
def implicit_preference_accuracy(model, rows):
    model.eval()
    correct = 0
    for row in tqdm(rows, desc="Quality DPO implicit accuracy"):
        prompt = quality_prompt(row)
        chosen_score = answer_logprob(model, prompt, row["chosen"], average=True)
        rejected_score = answer_logprob(model, prompt, row["rejected"], average=True)
        correct += int(float(chosen_score.cpu()) > float(rejected_score.cpu()))
    return correct / len(rows)

task4_metric = implicit_preference_accuracy(model, quality_test_data)
print(f"Task 4 Quality DPO implicit-preference accuracy = {task4_metric:.6f}")
print(f"Task 4 answer interval letter = {get_quality_letter(task4_metric)}")
